# W8｜从新项目到数据分析

- **项目与需求变更**
  - 第一轮：用一个短 Prompt 完成模拟、回归与结果检查。
  - 第二轮：增加实验，保留旧批次，检查实际改动。
  - 第三轮：打开工程设计，解释配置、随机种子和批次记录。
- **R／MATLAB 工作台**
  - 编辑器扩展、语言运行环境、Agent 各自负责什么？
  - Python 与 R 实际执行；MATLAB 只编辑与导读。
- **PostgreSQL 与外部数据**
  - 教师连接、查询、导出；学生用原项目继续分析。
  - 为 Longbridge 的身份、权限、请求和响应做准备。


## 如何使用本演示

- **教师**在这里展示需求、运行结果和设计对照；按顺序逐格运行，切到 VS Code 展示真正的项目文件。
- **学生**在自己的 VS Code 中与 Agent 开发，不需要在这份 Notebook 里填题。
- 第一轮只给目标和验收；第二轮再公布修改要求，第三轮才打开配置方案。
- 演示生成的文件在 `week-08/.generated/`；公开的备用数据不会被覆盖。默认不连接数据库。
- 课前完整运行一次；课堂前 **Restart Kernel**，从下一格开始。Notebook 的源文件保留为未运行版本。


In [ ]:
from pathlib import Path
import copy
import contextlib
import hashlib
import io
import json
import shutil
import subprocess
import sys
import tempfile

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Code, Image, Markdown, display

candidates = [Path.cwd(), Path.cwd() / "week-08"]
WEEK = next((p.resolve() for p in candidates if (p / "reference" / "simulate.py").exists()), None)
if WEEK is None:
    raise RuntimeError("请从课程仓库的 week-08/ 启动 Notebook，再重启 kernel。")
if str(WEEK) not in sys.path:
    sys.path.insert(0, str(WEEK))

from reference.simulate import generate_data, write_batch
from reference.analyze import load_data, fit_line, analyze_file

WORK = WEEK / ".generated"
DATA = WORK / "data"
OUTPUT = WORK / "outputs"
DATA.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
PURPLE, DARK, GREY = "#5e33bf", "#291344", "#77717d"
plt.rcParams.update({"figure.figsize": (10, 5), "font.size": 15,
                     "axes.titlesize": 19, "axes.labelsize": 16,
                     "axes.spines.top": False, "axes.spines.right": False})

def table(headers, rows):
    body = ["| " + " | ".join(map(str, headers)) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    body.extend("| " + " | ".join(map(str, row)) + " |" for row in rows)
    display(Markdown("\n".join(body)))

def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def run_analysis(input_path, prefix):
    # 命令行入口打印完整记录；演示页只展示当前需要的表和图。
    with contextlib.redirect_stdout(io.StringIO()):
        return analyze_file(input_path, prefix)

print("演示目录：", WEEK)
print("Python：", sys.executable)
print("NumPy：", np.__version__)


## 教学点一｜建立项目，并应对需求变化

### 第一轮：先让分析工作起来

我们模拟一批测量：`x` 是 `[0,10]` 上的 100 个等距点，`y=1+2x+noise`，噪声均值为 `0`、标准差为 `1`。

- **产物**：数据 CSV、散点与拟合直线、带名称的回归系数。
- **接口**：`analyze.py` 中的 `fit_line(x,y)` 返回 `(slope, intercept)`。
- **验收**：100 行、编号唯一、数据完整；已知无噪声例子算对；打开实际的图与结果。

这一轮实现方式自由。Agent 已经采用配置也可以，稍后用修改要求检验它。


### 给 Agent 的初始 Prompt

```text
在当前 uv 项目中做一个测量模拟和一元线性回归分析。
x 是 [0,10] 上 100 个等距点；用 np.random.default_rng(42) 一次生成独立正态噪声，
均值 0、标准差 1，y=1+2*x+noise。
保存 data/base-001.csv（sample_id,x,y），输出散点、拟合直线和带名称的回归系数。
analyze.py 提供 fit_line(x,y)，返回 (slope,intercept)，只能从输入数据估计。
先给最小计划，再实现并运行检查；告诉我实际运行命令。
```

学生在自己的 VS Code 运行。下面运行的是教师参考实现，先观察输入与结果，工程分工留到第三轮再看。


In [ ]:
# 先只关注本次实验取了哪些值，不急着规定学生如何组织参数。
baseline = dict(batch_id="base-001", n=100, seed=42, slope=2.0,
                intercept=1.0, noise_std=1.0, x_min=0.0, x_max=10.0)
base_path = DATA / "base-001.csv"
base_meta = write_batch(baseline, base_path)
base_result = run_analysis(base_path, OUTPUT / "base-001")
base_digest = digest(base_path)
table(["Input", "n", "Slope", "Intercept"], [[base_path.name, base_result["n"],
       f'{base_result["slope"]:.6f}', f'{base_result["intercept"]:.6f}']])
display(Image(filename=str(OUTPUT / "base-001.png")))


### 先检查实际生成了什么

- **结构**：有多少行？编号是否唯一？有没有缺失或非有限值？
- **设计**：`x` 的范围与间距是否符合要求？这里的 `x` 是固定设计，不是随机抽样。
- **现象**：点是否围绕上升直线分布？坐标轴是否拿反？数据有没有意外截断？

`程序结束了` 与 `这批数据符合实验设计` 是两件事。


In [ ]:
base_data = load_data(base_path)
x, y = base_data["x"], base_data["y"]
table(["Check", "Actual"], [
    ["Rows", len(x)], ["Unique sample_id", len(np.unique(base_data["sample_id"]))],
    ["Finite x and y", bool(np.isfinite(x).all() and np.isfinite(y).all())],
    ["x range", f"{x.min():.1f} .. {x.max():.1f}"],
    ["Equal x spacing", bool(np.allclose(np.diff(x), np.diff(x)[0]))],
    ["y mean", f"{y.mean():.4f}"], ["y sample SD", f"{y.std(ddof=1):.4f}"]])
print("前五行 [sample_id, x, y]：")
print(np.column_stack([base_data["sample_id"], x, y])[:5])


### 模拟阶段还能检查真实噪声

知道生成机制时，能算 `noise = y - (1 + 2*x)`。

- 看均值、样本标准差和直方图，判断有没有明显违背设定。
- 有限样本不要求均值恰好为 `0`、标准差恰好为 `1`；不设武断的单次通过阈值。
- `y` 还随 `x` 变化，`y` 的标准差不是噪声标准差；也不要求整列 `y` 的直方图呈正态。
- 图形和统计量是排查线索；看起来合理不能证明正态性、独立性，还要检查实际生成代码。


In [ ]:
true_noise = y - (baseline["intercept"] + baseline["slope"] * x)
table(["Quantity", "Design", "This sample"], [
    ["Noise mean", 0, f"{true_noise.mean():.4f}"],
    ["Noise SD", baseline["noise_std"], f"{true_noise.std(ddof=1):.4f}"]])
fig, ax = plt.subplots()
ax.hist(true_noise, bins=14, color=PURPLE, alpha=0.8, edgecolor="white")
ax.axvline(0, color=DARK, linestyle="--", label="Design mean = 0")
ax.set(xlabel="True noise", ylabel="Count", title="Does the generated noise look reasonable?")
ax.legend(); fig.tight_layout(); plt.show()


### 一元回归：真参数、估计值与预测值

$$y=\beta_0+\beta_1x+\varepsilon,\qquad \hat y=\hat\beta_0+\hat\beta_1x.$$

- `1,2` 是生成时设定的真参数；`intercept,slope` 是这批数据算出的估计。
- 最小二乘让残差平方和尽可能小。保留截距项；强制过原点是另一种模型。
- 真噪声是 `y-(1+2x)`；残差是 `y-y_hat`。实际应用通常不知道前者。
- 带截距的最小二乘残差均值接近零是代数性质，不能据此证明模型正确。


In [ ]:
slope, intercept = fit_line(x, y)
y_hat = intercept + slope * x
residual = y - y_hat
table(["Parameter", "Truth", "Estimate"], [
    ["Intercept", baseline["intercept"], f"{intercept:.6f}"],
    ["Slope", baseline["slope"], f"{slope:.6f}"]])
print(f"真实噪声均值 = {true_noise.mean():.6f}；拟合残差均值 = {residual.mean():.6f}")
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(x, y, color=PURPLE, alpha=0.65, s=28)
axes[0].plot(x, 1 + 2*x, "--", color=GREY, label="True line")
axes[0].plot(x, y_hat, color=DARK, label="OLS fit")
axes[0].set(xlabel="x", ylabel="y", title="Truth and estimate"); axes[0].legend()
axes[1].scatter(x, residual, color=PURPLE, alpha=0.7, s=28)
axes[1].axhline(0, color=DARK, linestyle="--")
axes[1].set(xlabel="x", ylabel="Residual", title="What structure remains?")
fig.tight_layout(); plt.show()


### 用确定答案检查计算

- 无噪声输入有确定答案；它检查计算与接口，不要求每批随机数据都得到真参数。
- 公开测试是 `checks/test_analysis.py`。学生复制到自己的 `tests/` 后运行 `uv run pytest -q`，核对测的是自己的 `analyze.py`。
- 期望值来自需求和已知直线，不调用被测函数生成“答案”。


In [ ]:
known_cases = [([0, 1, 2], [1, 3, 5], (2, 1)),
               ([0, 1, 2], [4, 3.5, 3], (-0.5, 4))]
for known_x, known_y, expected in known_cases:
    actual = fit_line(np.array(known_x, dtype=float), np.array(known_y, dtype=float))
    np.testing.assert_allclose(actual, expected, atol=1e-10)
    print(f"expected={expected}, actual={tuple(round(float(v), 10) for v in actual)}: PASS")


### 第二轮：新增实验，旧批次必须留下

现在继续修改同一个项目，不另起一份无法接续的程序。

- 保留 `base-001` 的数据与结果。
- `noise-002`：只把噪声标准差改为 `2`，仍用 `seed=42`。
- `seed-003`：回到噪声标准差 `1`，只把 seed 改为 `43`。
- 三批分别留下 CSV、图、系数与实际参数；拟合算法不随参数改写。


### 第二轮 Prompt 与验收

```text
保留 base-001 的数据和结果，新增两个批次：
noise-002：仅将噪声标准差改为 2，仍用 seed=42；
seed-003：回到噪声标准差 1，仅将 seed 改为 43。
其他设计不变。每批分别保存 CSV、图、回归系数与实际参数，明确它们的对应关系。
不要为新实验改写拟合算法；完成后重跑原检查，并确认旧批次未被覆盖。
```

**看实际文件**：三批齐全吗？基准数据有没有变？图和系数是否确实来自新数据？


In [ ]:
noisier = dict(baseline, batch_id="noise-002", noise_std=2.0)
new_seed = dict(baseline, batch_id="seed-003", seed=43)
batch_designs = [baseline, noisier, new_seed]
results = {"base-001": base_result}
for design in [noisier, new_seed]:
    batch_path = DATA / f'{design["batch_id"]}.csv'
    write_batch(design, batch_path)
    results[design["batch_id"]] = run_analysis(batch_path, OUTPUT / design["batch_id"])
table(["Batch", "n", "Seed", "Noise SD", "Slope", "Intercept"], [
    [c["batch_id"], c["n"], c["seed"], c["noise_std"],
     f'{results[c["batch_id"]]["slope"]:.6f}', f'{results[c["batch_id"]]["intercept"]:.6f}']
    for c in batch_designs])


### 一直写入 `results.csv`，留下的是什么？

- 同一个输出路径通常只留下最后一批；文件还在，不代表前一批还在。
- 下面只在临时目录演示覆盖，不碰公开数据或真实项目产物。


In [ ]:
with tempfile.TemporaryDirectory(prefix="w8-overwrite-") as tmp:
    same_path = Path(tmp) / "results.csv"
    write_batch(baseline, same_path)
    first_content = same_path.read_bytes()
    write_batch(new_seed, same_path)
    last_meta = json.loads(same_path.with_suffix(".meta.json").read_text())
    print("输出路径相同：results.csv")
    print("内容是否仍为第一批：", same_path.read_bytes() == first_content)
    print("现在对应的批次：", last_meta["batch_id"])


### 用产物核对修改，而不是只听“完成了”

- 第一批的数据内容保持；新增批次使用不同文件名。
- 从实际 CSV 读入数据，再检查新旧批次确实不同。
- 最终成品说明项目当前做到了什么，不能单凭完成速度判断学习态度。


In [ ]:
assert digest(base_path) == base_digest
noise_data = load_data(DATA / "noise-002.csv")
seed_data = load_data(DATA / "seed-003.csv")
assert not np.array_equal(base_data["y"], noise_data["y"])
assert not np.array_equal(base_data["y"], seed_data["y"])
assert all((OUTPUT / f'{c["batch_id"]}-fit.csv').exists() for c in batch_designs)
print("PASS：基准 CSV 保持，两个新批次独立保存，三批均有分析结果。")
print("本格比较的是实际内容，不以文件名或时间戳代替检查。")


### 第三轮：这些工程设计解决了什么问题？

经历修改以后，再看参数与程序如何分工：

- 参数散落在代码里，每次改实验需要找出所有相关位置。
- 参数集中在一份配置里，生成和分析代码就能保持职责稳定。
- **配置快照**记录生成当时实际用了什么；后来改配置，不改旧数据的记录。
- 下面才打开参考项目的配置方案。学生已有合理结构可以继续维护。


In [ ]:
config_path = WEEK / "reference" / "configs" / "base-001.json"
config = json.loads(config_path.read_text())
display(Code(filename=str(config_path), language="json"))
assert all(config[k] == baseline[k] for k in baseline)


### 配置、环境与 CSV 批次各自说明什么

- **JSON／TOML**：两者都能承载配置，本节参考使用 JSON，不要求同时维护两套。
  - `pyproject.toml` 声明项目与依赖；实验配置记录样本量、seed、系数、噪声和范围。
  - `noise_std` 是标准差，与 `y` 同单位；取 `2` 时方差是 `4`。检查函数参数的实际含义。
- **CSV 批次命名**：`base-001.csv`、`noise-002.csv`、`seed-003.csv` 帮助区分批次。
  - 名字不证明内容正确；批次命名也不是 Git 版本控制。
  - Git 留代码与配置历史；数据和结果按批次留存，避免相互覆盖。


In [ ]:
# 真正运行参考项目的命令行入口：只换配置和输出位置。
subprocess.run([sys.executable, str(WEEK / "reference" / "simulate.py"),
                "--config", str(config_path), "--out", str(DATA / "config-demo.csv")], check=True)
config_data = load_data(DATA / "config-demo.csv")
np.testing.assert_allclose(config_data["x"], x)
np.testing.assert_allclose(config_data["y"], y)
print("配置入口与本轮基准使用相同设计；数据内容核对通过。")


### 时间戳、批次标签和配置快照

- 同名 `.meta.json` 留下批次、UTC 生成时间、实际配置和软件版本。
- 时间戳回答何时生成，批次标签便于定位；二者都不能替代内容核对。
- 相同设计重跑时，CSV 可以相同，生成时间却不同；不要比较整份元数据来判断随机抽样是否一致。
- 本例没有逐条观测时间。`sample_id` 是编号，批量生成时间也不是每条记录的真实观测时间。


In [ ]:
saved_meta = json.loads(base_path.with_suffix(".meta.json").read_text())
print(json.dumps(saved_meta, ensure_ascii=False, indent=2))
config["noise_std"] = 9.0  # 只修改内存中的当前设计，不改旧批次文件。
assert saved_meta["config"]["noise_std"] == 1.0
print("当前内存参数改为 9，旧批次的配置快照仍为 1。")


### seed 固定了什么，没有固定什么？

- 每次模拟开始创建一次生成器，随后持续取样。
- 在循环内反复重设 seed，可能让每个点都得到同一个噪声。
- 同一生成代码、软件环境和随机调用顺序下谈重现；跨语言写相同 seed，不代表同一组数据。
- `n=100` 是一批中的 100 条观测；重新使用相同 seed 不是独立重复实验。


In [ ]:
wrong_noise = np.array([np.random.default_rng(42).normal() for _ in range(6)])
rng = np.random.default_rng(42)
right_noise = rng.normal(size=6)
table(["Method", "Six draws"], [["Reset seed in each iteration", np.round(wrong_noise, 3).tolist()],
                                  ["One generator, continuous draws", np.round(right_noise, 3).tolist()]])


### 两种“改噪声”的比较不同

- 固定 `x`、seed 和调用顺序，只放大标准差，相当于放大同一组标准正态抽样。
- 这个一元最小二乘例子中，系数相对真值的偏差也随噪声尺度缩放。
- 另换 seed 时，抽样本身变了；不能要求某一次误差必然更大。


In [ ]:
base_coef = np.array([results["base-001"]["slope"], results["base-001"]["intercept"]])
noise_coef = np.array([results["noise-002"]["slope"], results["noise-002"]["intercept"]])
seed_coef = np.array([results["seed-003"]["slope"], results["seed-003"]["intercept"]])
truth = np.array([2.0, 1.0])
np.testing.assert_allclose(noise_coef-truth, 2*(base_coef-truth), atol=1e-10)
table(["Batch", "Slope error", "Intercept error"], [
    [name, f"{(coef-truth)[0]:.6f}", f"{(coef-truth)[1]:.6f}"]
    for name, coef in [("base-001", base_coef), ("noise-002", noise_coef), ("seed-003", seed_coef)]])


### Agent 帮助排错：先给证据，再改实现

函数承诺返回 `(slope,intercept)`，但某个版本把顺序颠倒了。

```text
预期 (2,1)，实际 (1,2)。沿测试、函数返回值与调用位置定位，
先给依据再做最小修复，不修改预期答案。
```

核对函数承诺、库函数的输出顺序、调用者如何接收。修复后重跑原检查和另一份输入。


In [ ]:
known_x = np.array([0., 1., 2.])
known_y = np.array([1., 3., 5.])
correct = fit_line(known_x, known_y)
wrong_order = tuple(reversed(correct))
try:
    np.testing.assert_allclose(wrong_order, (2, 1), atol=1e-10)
except AssertionError:
    print("检查正确地拒绝了顺序颠倒的版本：", tuple(round(float(v), 6) for v in wrong_order))
np.testing.assert_allclose(correct, (2, 1), atol=1e-10)
np.testing.assert_allclose(fit_line(known_x, np.array([4., 3.5, 3.])), (-0.5, 4), atol=1e-10)
print("按接口接收结果后，原检查和变式均通过。")


## 教学点二｜配置 R／MATLAB 的 VS Code 工作台

### R：扩展与运行环境各负责什么？

- 在 VS Code 打开项目，确认正在使用本机还是 WSL 工作区。
- 确认 `Rscript --version`，安装 **R** 扩展，按 Guide 配置 `languageserver`。
- 打开 `.R` 文件，查看编辑提示、R 终端和运行结果。
- 扩展提供编辑与交互；R 运行环境真正执行代码。`uv` 不安装 R。
- Python 可以输出 CSV，再交给 R 画图；本节不做语言相互调用。

[VS Code 的 R 工作台说明](https://code.visualstudio.com/docs/languages/r)


### 让 Agent 帮助处理同一分析

```text
读取 data/base-001.csv，用基础 R 拟合 y ~ x，打印 summary(model)，
保存散点和拟合直线。写成 plot.R，接收输入 CSV 与输出图片两个参数。
先解释，再给运行命令，不新增 R 包。
```

在 VS Code 打开 `reference/plot.R`，指认输入、模型、系数和输出；随后实际运行。


In [ ]:
rscript = shutil.which("Rscript")
if rscript is None:
    print("本机没有 Rscript：本格跳过实际执行。按 Guide 配好 R 后再重跑，不把示例文字当运行结果。")
else:
    r_output = OUTPUT / "base-001-r.png"
    r_run = subprocess.run([rscript, "--vanilla", str(WEEK / "reference" / "plot.R"),
                            str(base_path), str(r_output)], capture_output=True, text=True, timeout=60)
    if r_run.returncode != 0:
        print("R 尚未运行成功，请在终端按实际错误排查：")
        print(r_run.stderr[-2500:])
    else:
        print(r_run.stdout)
        display(Image(filename=str(r_output)))


### 阅读 R 的结果

- `Estimate`：截距、斜率的估计；按名称辨认，不假定不同语言的排列顺序相同。
- `Std. Error`：模型假设下，系数估计的不确定性。
- `Residual standard error`：本例为 `sqrt(SSE/(n-2))`；`np.std(residual, ddof=1)` 用 `n-1`，分母不同。
- Python 与 R 读取**同一份 CSV**，再对照系数。高 `R²`、显著 p 值不能替代程序检查。
- 残差图中是否仍有曲线或随 `x` 扩大的波动，是继续检查的线索，不是一次图形观察的定论。


### MATLAB：建立编辑与分析工作台

- 在 VS Code 安装 **MathWorks 官方 MATLAB 扩展**，打开下面的 `.m` 文件。
- 区分三层：扩展负责编辑支持，Agent 帮助解释与修改，MATLAB 本体和授权决定能否运行。
- 本节只展示代码阅读和修改，不安装 MATLAB 本体、不执行示例。

```text
逐段指出读数据、拟合和绘图在哪里；将系数改成带名称的输出。
解释后再修改，不执行 MATLAB。
```

[MathWorks 官方扩展说明](https://github.com/mathworks/MATLAB-extension-for-vscode)


In [ ]:
display(Code(filename=str(WEEK / "reference" / "analysis.m"), language="matlab"))
print("本格仅展示源代码，未启动 MATLAB，也未验证 MATLAB 的执行结果。")


## 教学点三｜连接数据库，接续外部数据

### 客户端、服务端和连接配置

```text
本机 Python 客户端 → PostgreSQL 服务 → 查询结果 → CSV → 原分析程序
```

- 主机与端口定位服务；数据库与用户决定连接到哪里、以谁的身份访问。
- 数据库连接不同于 SSH 登录服务器。
- 连接配置与凭证由教师本人准备；Agent 可帮助写调用代码，不需要看到真实密码。
- 学生领取 CSV 和同名元数据继续分析，不安装数据库、不领取连接凭证。


### 这次查询要取哪些记录？

```sql
SELECT sample_id, x, y
FROM w08_demo.measurements
WHERE dataset = 'calibration'
ORDER BY sample_id;
```

- 取三列，选择校准批次，按编号排序。
- 校准批次原样保存基准数据；换存放位置不是重新抽样。
- **默认使用合成备用文件**。只有教师按 `reference/db/README.md` 配好只读连接后，才把下一格的开关改为 `True`。


In [ ]:
RUN_DATABASE = False  # 只有课前已配好服务 w08_demo 才开启；不会从 Notebook 提供密码。
USE_LIVE_DATABASE = False
if RUN_DATABASE:
    try:
        for dataset in ("calibration", "application"):
            filename = "db-calibration.csv" if dataset == "calibration" else "application.csv"
            subprocess.run([sys.executable, str(WEEK / "reference" / "fetch_data.py"),
                            "--dataset", dataset, "--out", str(DATA / filename)],
                           check=True, capture_output=True, text=True, timeout=15)
        USE_LIVE_DATABASE = True
        print("本次实际调用 PostgreSQL，并已导出两批数据。")
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
        print("本次数据库取数未完成；不投影连接错误细节，改用公开合成备用文件。")

if USE_LIVE_DATABASE:
    calibration_path = DATA / "db-calibration.csv"
    application_path = DATA / "application.csv"
    source_label = "本次实际 PostgreSQL 调用"
else:
    calibration_path = WEEK / "data" / "calibration.csv"
    application_path = WEEK / "data" / "application.csv"
    source_label = "synthetic_fixture：合成备用文件，本次未完成现场数据库取数"
print(source_label)
print(json.dumps(json.loads(calibration_path.with_suffix(".meta.json").read_text()),
                 ensure_ascii=False, indent=2))


In [ ]:
calibration = load_data(calibration_path)
base_order = np.argsort(base_data["sample_id"])
db_order = np.argsort(calibration["sample_id"])
np.testing.assert_array_equal(base_data["sample_id"][base_order], calibration["sample_id"][db_order])
np.testing.assert_allclose(base_data["x"][base_order], calibration["x"][db_order])
np.testing.assert_allclose(base_data["y"][base_order], calibration["y"][db_order])
calibration_result = run_analysis(calibration_path, OUTPUT / "calibration")
np.testing.assert_allclose([calibration_result["slope"], calibration_result["intercept"]],
                           [base_result["slope"], base_result["intercept"]], atol=1e-10)
print("校准检查通过：同一批记录，数据与回归结果一致。")
print("来源：", source_label)


### 应用：分析另一批模拟实验记录

- 把 `application.csv` 与 `application.meta.json` 交给学生，放进自己的项目。
- 先检查来源、字段、行数、缺失、范围和散点，再沿用原分析估计参数、检查残差。

```text
请检查这份新数据能否交给现有分析，说明输入和来源；沿用拟合方法，
输出这一批的图、系数与残差检查，不把结果写死为原模拟参数。
```

**验收**：实际用了哪批、多少行？结果保存在哪里？原批次是否还在？


In [ ]:
application_meta = json.loads(application_path.with_suffix(".meta.json").read_text())
print("来源：", source_label)
print(json.dumps(application_meta, ensure_ascii=False, indent=2))
application_result = run_analysis(application_path, OUTPUT / "application")
table(["Input", "n", "Slope", "Intercept"], [[application_path.name, application_result["n"],
       f'{application_result["slope"]:.6f}', f'{application_result["intercept"]:.6f}']])
display(Image(filename=str(OUTPUT / "application.png")))
display(Image(filename=str(OUTPUT / "application-residuals.png")))


### 没有真参数时，我们能判断什么？

- 取数成功不等于数据正确，程序正常结束也不等于解释合理。
- 没有生成配置时，只能计算估计残差，不把它称为真实噪声。
- 已知答案测试核对确定性计算；批次记录说明实际输入；统计诊断提供模型检查线索。
- 完成分析后，再看教师保存的生成设计。


In [ ]:
application_truth = dict(n=80, seed=7, slope=-0.5, intercept=4.0,
                         noise_std=0.5, x_min=0.0, x_max=10.0)
table(["Parameter", "Generation", "Estimate"], [
    ["Slope", application_truth["slope"], f'{application_result["slope"]:.6f}'],
    ["Intercept", application_truth["intercept"], f'{application_result["intercept"]:.6f}']])
print("教师使用合成数据，所以能在这里揭示真值；真实应用通常没有这份答案。")


### 为 Longbridge 接入做准备

- **服务地址**：去哪里发送请求？
- **认证**：以谁的身份？
- **授权**：允许做什么？
- **请求与响应**：这次实际上拿到了什么？

下一步更换数据服务，仍先检查字段、数量、时间与缺失。本节不安装 Longbridge，不进行交易授权。


### 本次课回顾

- 项目应对需求变化：新增实验，保留旧批次，图、系数、参数与数据对应。
- 工作台接入语言：编辑器扩展、运行环境、Agent 分工明确；Python 与 R 通过 CSV 交接。
- 外部数据接续分析：连接、取数、数据检查与模型判断分别留下证据。
- 用 Git 保存代码、配置、测试、README 和环境记录；CSV 与元数据按批次保留，连接凭证不进入仓库。


### 提前完成：再接一个修改要求

增加按 `x` 区间分析的能力，先筛选再拟合，并记录实际使用的样本数。

- 已知 `x=[0,1,2,3]、y=[1,3,5,100]`，只取 `0≤x≤2` 时应得到斜率 `2`、截距 `1`。
- 不设置区间时，保留原来的分析行为。
- 先写检查再修改，保存这一次输入、结果与改动。实现路径由学生与 Agent 自行决定。
